In [ ]:
import pandas as pd
import notebook_config
from extract import tabela_pagamentos

pagamentos = tabela_pagamentos()

In [ ]:
# conhecer a estrutura
print("PAGAMENTOS")
print("Linhas e colunas:", pagamentos.shape)
print(pagamentos.head())
pagamentos.info()

In [ ]:
# nulos por coluna
print("Nulos em Pagamentos:")
print(pagamentos.isna().sum())

# resumo da auditoria
resumo = pd.DataFrame({
    "Tabela": ["Pagamentos"],
    "Linhas": [len(pagamentos)],
    "Pedidos distintos": [
        pagamentos["order_id"].nunique()
    ],
    "Repetições completas além da primeira": [
        pagamentos.duplicated().sum()
    ]
})

print(resumo)

In [ ]:
# verificar os valores numéricos
print("Estatísticas de Pagamentos:")
print(pagamentos.describe())

print("\nDistribuição das formas de pagamento:")
print(pagamentos["payment_type"].value_counts(dropna=False))

In [ ]:
# Seleciona os registros que precisam de investigação
valor_zero = pagamentos[pagamentos["payment_value"] == 0]

parcelas_zero = pagamentos[pagamentos["payment_installments"] == 0]

tipo_indefinido = pagamentos[
    pagamentos["payment_type"] == "not_defined"
]

print("PAGAMENTOS COM VALOR ZERO:")
print(valor_zero.to_string(index=False))

print("\nPAGAMENTOS COM ZERO PARCELAS:")
print(parcelas_zero.to_string(index=False))

print("\nPAGAMENTOS COM TIPO NÃO DEFINIDO:")
print(tipo_indefinido.to_string(index=False))

In [ ]:
# Identifica os pedidos que possuem algum pagamento de valor zero
pedidos_com_zero = valor_zero["order_id"].unique()

# Recupera todos os pagamentos desses pedidos
pagamentos_desses_pedidos = pagamentos[
    pagamentos["order_id"].isin(pedidos_com_zero)
]

# Resume os pagamentos por pedido
resumo_pedidos_zero = pagamentos_desses_pedidos.groupby("order_id").agg(
    quantidade_registros=("payment_value", "size"),
    valor_total_pago=("payment_value", "sum")
)

print(resumo_pedidos_zero.to_string())

chaves_repetidas = pagamentos.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

print("Repetições de pedido + sequência do pagamento:", chaves_repetidas)